# Imports

In [1]:
import os
import pickle
import re
import shutil
import sys
sys.path.append(os.path.dirname(os.getcwd()))
from itertools import product

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from matplotlib.backends.backend_pdf import PdfPages
from tools import load_npy, load_yaml_as_df, load_pkl, exist_metric, exist_stf_metric, inverse_stf_metrics, keep_split, is_full_group, load_metric_from_log

plt.style.use('default')
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
plt.rc('font', family='Arial')
matplotlib.rcParams['mathtext.fontset'] = 'stix'
matplotlib.rcParams['font.size'] = 10

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# Baselines local

## load data

In [24]:
root = '/data/home/Licheng/workspace/ProjDF-Meta/results_ML3/baselines'
exp_dirs = os.listdir(root)
exp_dirs = [os.path.join(root, exp_dir) for exp_dir in exp_dirs]

params = ['model', 'pred_len', 'data_id', 'learning_rate', 'batch_size', 'patience', 'individual', 'train_epochs', 'lradj']
metric_names = ['mse', 'mae']

df = []
for exp_dir in exp_dirs:
    runned, setting_dir = exist_metric(exp_dir)
    if not runned:
        continue

    config = load_yaml_as_df(os.path.join(setting_dir, 'config.yaml'))
    metric = load_npy(os.path.join(setting_dir, 'metrics.npy'))
    result = config[params]
    model, individual = config['model'].values[0], config['individual'].values[0]
    if model == 'DLinear' and individual:
        result['model'].values[0] = 'DLinear_Ind'
    result.loc[:, metric_names] = metric[1], metric[0]
    result.loc[:, ['exp_dir']] = exp_dir
    df.append(result)

df = pd.concat(df, ignore_index=True)
df.sort_values(by=['model', 'data_id', 'pred_len'], inplace=True)

# save_root = '/data/home/Licheng/workspace/TSF-CCA/stats_CCA'
# os.makedirs(save_root, exist_ok=True)
# df.to_csv(f"{save_root}/baseline_local.csv", index=False)

df.head(4)

,model,pred_len,data_id,learning_rate,batch_size,patience,individual,train_epochs,lradj,mse,mae,exp_dir
4,xPatch,96,ECL,0.0005,16,10,0,100,sigmoid,0.159165,0.250191,/data/home/Licheng/workspace/ProjDF-Meta/resul...
1,xPatch,192,ECL,0.0005,16,10,0,100,sigmoid,0.169169,0.259204,/data/home/Licheng/workspace/ProjDF-Meta/resul...
29,xPatch,336,ECL,0.0005,16,10,0,100,sigmoid,0.184928,0.275742,/data/home/Licheng/workspace/ProjDF-Meta/resul...
18,xPatch,720,ECL,0.0005,16,10,0,100,sigmoid,0.224763,0.310016,/data/home/Licheng/workspace/ProjDF-Meta/resul...


## analysis

In [25]:
min_mode = 'each'

df2 = df.copy()

columns = ['model', 'data_id', 'learning_rate', 'batch_size', 'patience', 'individual', 'train_epochs']
if min_mode == 'group':
    mse_mean = df2.groupby(columns)['mse'].mean().reset_index()
    idx = mse_mean.groupby(['model', 'data_id'])['mse'].idxmin()
    best_model = mse_mean.loc[idx]
    df2 = df2.merge(best_model[columns], on=columns, how='inner')
elif min_mode == 'each':
    min_mse_idx = df2.groupby(['model', 'data_id', 'pred_len'])['mse'].idxmin()
    df2 = df2.loc[min_mse_idx]

# save_root = '/data/home/Licheng/workspace/TSF-CCA/stats_CCA'
# df2.to_csv(f'{save_root}/baselines_local_params.csv', index=False)

df2 = df2[['model', 'pred_len', 'data_id', 'mse', 'mae']]

dst_order = ['ETTm1', 'ETTm2', 'ETTh1', 'ETTh2', 'ECL', 'Traffic', 'Weather', 'PEMS03', 'PEMS08']
df2['data_id'] = pd.Categorical(df2['data_id'], categories=dst_order, ordered=True)

model_order = ['xPatch', 'PDF', 'CycleNet', 'PatchTST', 'DLinear', 'Fredformer', 'iTransformer', 'FreTS', 'TimesNet', 'MICN', 'TiDE']
df2 = df2[df2['model'].isin(model_order)]
df2['model'] = pd.Categorical(df2['model'], categories=model_order, ordered=True)

df2_avg = df2.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()
df2_avg['pred_len'] = 'Avg'
df2 = pd.concat([df2, df2_avg]).reset_index(drop=True)

df2.sort_values(by=['model', 'data_id', 'pred_len'], inplace=True)

df2 = df2.set_index(['data_id', 'pred_len', 'model']).unstack('model').swaplevel(axis=1)
columns = []
for model in df2.columns.levels[0]:
    columns.append((model, 'mse'))
    columns.append((model, 'mae'))
df2 = df2[columns]

# df2.to_excel(f'{save_root}/baselines_local.xlsx')
# df2.to_csv(f'{save_root}/baselines_local.csv')
df2

/tmp/ipykernel_4170202/1365959705.py:27: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df2_avg = df2.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()


model               xPatch           PDF     CycleNet     PatchTST      \
                       mse       mae mse mae      mse mae      mse mae   
data_id pred_len                                                         
ETTm1   96        0.332835  0.370666 NaN NaN      NaN NaN      NaN NaN   
        192       0.361721  0.383598 NaN NaN      NaN NaN      NaN NaN   
        336       0.398729  0.410044 NaN NaN      NaN NaN      NaN NaN   
        720       0.458187  0.448617 NaN NaN      NaN NaN      NaN NaN   
        Avg       0.387868  0.403232 NaN NaN      NaN NaN      NaN NaN   
ETTm2   96        0.177255  0.261752 NaN NaN      NaN NaN      NaN NaN   
        192       0.240805  0.302909 NaN NaN      NaN NaN      NaN NaN   
        336       0.301417  0.344264 NaN NaN      NaN NaN      NaN NaN   
        720       0.402171  0.399906 NaN NaN      NaN NaN      NaN NaN   
        Avg       0.280412  0.327208 NaN NaN      NaN NaN      NaN NaN   
ETTh1   96        0.379179  0.401771 NaN NaN      NaN NaN      NaN NaN   
        192       0.440351  0.435595 NaN NaN      NaN NaN      NaN NaN   
        336       0.484914  0.453432 NaN NaN      NaN NaN      NaN NaN   
        720       0.494353  0.480334 NaN NaN      NaN NaN      NaN NaN   
        Avg       0.449699  0.442783 NaN NaN      NaN NaN      NaN NaN   
ETTh2   96        0.295351  0.345930 NaN NaN      NaN NaN      NaN NaN   
        192       0.370672  0.395091 NaN NaN      NaN NaN      NaN NaN   
        336       0.425151  0.436900 NaN NaN      NaN NaN      NaN NaN   
        720       0.421763  0.444647 NaN NaN      NaN NaN      NaN NaN   
        Avg       0.378234  0.405642 NaN NaN      NaN NaN      NaN NaN   
ECL     96        0.159165  0.250191 NaN NaN      NaN NaN      NaN NaN   
        192       0.169169  0.259204 NaN NaN      NaN NaN      NaN NaN   
        336       0.184928  0.275742 NaN NaN      NaN NaN      NaN NaN   
        720       0.224763  0.310016 NaN NaN      NaN NaN      NaN NaN   
        Avg       0.184506  0.273788 NaN NaN      NaN NaN      NaN NaN   
Traffic 96        0.482954  0.301660 NaN NaN      NaN NaN      NaN NaN   
        192       0.490971  0.301667 NaN NaN      NaN NaN      NaN NaN   
        Avg       0.486962  0.301663 NaN NaN      NaN NaN      NaN NaN   
Weather 96        0.164681  0.210172 NaN NaN      NaN NaN      NaN NaN   
        192       0.208169  0.248611 NaN NaN      NaN NaN      NaN NaN   
        336       0.264768  0.289887 NaN NaN      NaN NaN      NaN NaN   
        720       0.343657  0.341714 NaN NaN      NaN NaN      NaN NaN   
        Avg       0.245319  0.272596 NaN NaN      NaN NaN      NaN NaN   
PEMS03  12        0.079372  0.187961 NaN NaN      NaN NaN      NaN NaN   
        24        0.120121  0.232217 NaN NaN      NaN NaN      NaN NaN   
        36        0.163120  0.271434 NaN NaN      NaN NaN      NaN NaN   
        48        0.207312  0.307271 NaN NaN      NaN NaN      NaN NaN   
        Avg       0.142482  0.249721 NaN NaN      NaN NaN      NaN NaN   
PEMS08  12        0.091401  0.196122 NaN NaN      NaN NaN      NaN NaN   
        24        0.138754  0.240838 NaN NaN      NaN NaN      NaN NaN   
        36        0.189931  0.282044 NaN NaN      NaN NaN      NaN NaN   
        48        0.234162  0.316696 NaN NaN      NaN NaN      NaN NaN   
        Avg       0.163562  0.258925 NaN NaN      NaN NaN      NaN NaN   

model            DLinear     Fredformer     iTransformer     FreTS      \
                     mse mae        mse mae          mse mae   mse mae   
data_id pred_len                                                         
ETTm1   96           NaN NaN        NaN NaN          NaN NaN   NaN NaN   
        192          NaN NaN        NaN NaN          NaN NaN   NaN NaN   
        336          NaN NaN        NaN NaN          NaN NaN   NaN NaN   
        720          NaN NaN        NaN NaN          NaN NaN   NaN NaN   
        Avg          NaN NaN        NaN NaN          NaN NaN   NaN NaN   
ETT

# Finetune

## load data

In [4]:
root = '/data/home/Licheng/workspace/ProjDF-Meta/results_META/finetune'
exp_dirs = os.listdir(root)
exp_dirs = [os.path.join(root, exp_dir) for exp_dir in exp_dirs]


params = ['model', 'pred_len', 'data_id', 'learning_rate', 'inner_lr', 'meta_lr', 'rec_lambda', 'auxi_lambda', 'reg_lambda', 'lradj', 'train_epochs', 'patience', 'batch_size', 'auxi_batch_size', 'fixed_step', 'meta_inner_steps', 'overlap_ratio', 'num_tasks', 'max_norm', 'auxi_loss', 'first_order', 'dropout', 'cycle']
metric_names = ['mse', 'mae', 'cov']

df = []
for exp_dir in exp_dirs:
    runned, setting_dir = exist_metric(exp_dir)
    if not runned:
        continue

    config = load_yaml_as_df(os.path.join(setting_dir, 'config.yaml'))
    metric = load_npy(os.path.join(setting_dir, 'metrics.npy'))
    result = config[params]
    if len(metric) == 6:
        log_metrics = load_metric_from_log(os.path.join(exp_dir, 'result_long_term_forecast.txt'))
        cov_loss = log_metrics['cov']
        result.loc[:, metric_names] = metric[1], metric[0], cov_loss
    else:
        result.loc[:, metric_names] = metric[1], metric[0], metric[2]
    df.append(result)

df = pd.concat(df, ignore_index=True)

df.sort_values(by=['model', 'data_id', 'pred_len'], inplace=True)

save_root = '/data/home/Licheng/workspace/ProjDF-Meta/stats_META'
os.makedirs(save_root, exist_ok=True)
df.to_csv(f"{save_root}/finetune_all_results.csv", index=False)

df.head(4)

,model,pred_len,data_id,learning_rate,inner_lr,meta_lr,rec_lambda,auxi_lambda,reg_lambda,lradj,train_epochs,patience,batch_size,auxi_batch_size,fixed_step,meta_inner_steps,overlap_ratio,num_tasks,max_norm,auxi_loss,first_order,dropout,cycle,mse,mae,cov
0,TQNet,96,ETTh1,0.0005,0.0005,0.00025,1.0,0.0,0.0,type1,30,5,32,256,300,1,0.15,3,1.0,MAE,1,0.5,24,0.370129,0.383088,0.382454
8,TQNet,96,ETTh1,0.0020,0.0020,0.00100,1.0,0.0,0.0,type1,30,5,32,256,300,1,0.15,3,1.0,MAE,1,0.5,24,0.365848,0.382437,0.380029
34,TQNet,96,ETTh1,0.0020,0.0020,0.00100,1.0,0.0,0.0,type1,30,5,32,256,700,1,0.15,3,1.0,MAE,1,0.5,24,0.365936,0.382489,0.381485
96,TQNet,96,ETTh1,0.0010,0.0010,0.00050,1.0,0.0,0.0,type1,30,5,32,256,700,1,0.15,3,1.0,MAE,1,0.5,24,0.368454,0.382417,0.381880


## pre-load

In [3]:
save_root = '/data/home/Licheng/workspace/ProjDF-Meta/stats_META'
df = pd.read_csv(f'{save_root}/finetune_all_results.csv')

df.head(4)

,model,pred_len,data_id,learning_rate,inner_lr,meta_lr,rec_lambda,auxi_lambda,reg_lambda,lradj,train_epochs,patience,batch_size,auxi_batch_size,fixed_step,meta_inner_steps,overlap_ratio,num_tasks,max_norm,auxi_loss,first_order,dropout,cycle,mse,mae
0,TQNet,96,ETTh1,0.0005,0.0005,0.00025,1.0,0.0,0.0,type1,30,5,32,256,300,1,0.15,3,1.0,MAE,1,0.5,24,0.370129,0.383088
1,TQNet,96,ETTh1,0.0020,0.0020,0.00100,1.0,0.0,0.0,type1,30,5,32,256,300,1,0.15,3,1.0,MAE,1,0.5,24,0.365848,0.382437
2,TQNet,96,ETTh1,0.0020,0.0020,0.00100,1.0,0.0,0.0,type1,30,5,32,256,700,1,0.15,3,1.0,MAE,1,0.5,24,0.365936,0.382489
3,TQNet,96,ETTh1,0.0010,0.0010,0.00050,1.0,0.0,0.0,type1,30,5,32,256,700,1,0.15,3,1.0,MAE,1,0.5,24,0.368454,0.382417


## pre-analysis

In [ ]:
df_sort = df.sort_values(by=['model', 'data_id', 'pred_len', 'mse'])[['data_id', 'pred_len', 'exp_dir', 'mse']].groupby(['data_id', 'pred_len']).head(5)

columns = ['model', 'pred_len', 'data_id', 'mse', 'mae', 'learning_rate', 'inner_lr', 'rank_ratio', 'align_type', 'individual', 'fixed_epoch', 'fixed_step', 'learn_x_proj', 'learn_y_proj', 'lradj', 'train_epochs', 'patience', 'proj_init', 'reg_lambda', 'batch_size', 'pre_norm', 'identity_direction']

df_sort = df.sort_values(by=['model', 'data_id', 'pred_len', 'mse']).groupby(['data_id', 'pred_len']).head(5)
df_sort = df_sort[df_sort['data_id'].isin(['ETTh2'])]
df_sort[columns]

for i, row in df_sort.iterrows():
    exp_dir = row['exp_dir']
    mse = row['mse']
    print(f"Data ID: {row['data_id']}, Pred Len: {row['pred_len']}, MSE: {mse:.4f}")
    print(exp_dir)

## analysis

In [5]:
min_mode = 'sum'

df2 = df.copy()

columns = ['model', 'data_id', 'learning_rate', 'alpha', 'rank_ratio', 'align_type', 'individual']
if min_mode == 'group':
    mse_mean = df2.groupby(columns)['mse'].mean().reset_index()
    idx = mse_mean.groupby(['model', 'data_id'])['mse'].idxmin()
    best_model = mse_mean.loc[idx]
    df2 = df2.merge(best_model[columns], on=columns, how='inner')
elif min_mode == 'each':
    min_mse_idx = df2.groupby(['model', 'data_id', 'pred_len'])['mse'].idxmin()
    df2 = df2.loc[min_mse_idx]
elif min_mode == 'sum':
    # 新增：mse+mae最小
    df2['sum_error'] = df2['mse'] + df2['mae']
    min_sum_idx = df2.groupby(['model', 'data_id', 'pred_len'])['sum_error'].idxmin()
    df2 = df2.loc[min_sum_idx]
    df2 = df2.drop(columns=['sum_error'])

columns = ['model', 'pred_len', 'data_id', 'mse', 'mae', 'cov', 'learning_rate', 'inner_lr', 'meta_lr', 'auxi_loss', 'rec_lambda', 'auxi_lambda', 'reg_lambda', 'lradj', 'train_epochs', 'patience', 'batch_size', 'auxi_batch_size', 'fixed_step', 'meta_inner_steps', 'overlap_ratio', 'num_tasks', 'max_norm', 'first_order', 'dropout', 'cycle']
df2 = df2[columns]

dst_order = ['ETTm1', 'ETTm2', 'ETTh1', 'ETTh2', 'ECL', 'Traffic', 'Weather', 'PEMS03', 'PEMS08']
df2['data_id'] = pd.Categorical(df2['data_id'], categories=dst_order, ordered=True)

model_order = ['TQNet']
df2['model'] = pd.Categorical(df2['model'], categories=model_order, ordered=True)
df2.sort_values(by=['model', 'data_id', 'pred_len'], inplace=True)

# print(df2)
df2_avg = df2.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()
df2_avg['pred_len'] = 'Avg'

df2 = pd.concat([df2, df2_avg]).reset_index(drop=True)

# pl_order = [96, 192, 336, 720, 'Avg']
# df2['pred_len'] = pd.Categorical(df2['pred_len'], categories=pl_order, ordered=True)

df2.sort_values(by=['data_id', 'model', 'pred_len'], inplace=True)
df2.dropna(inplace=True, thresh=5)
df2

/tmp/ipykernel_3015649/1773224260.py:32: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df2_avg = df2.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()


,model,pred_len,data_id,mse,mae,cov,learning_rate,inner_lr,meta_lr,auxi_loss,rec_lambda,auxi_lambda,reg_lambda,lradj,train_epochs,patience,batch_size,auxi_batch_size,fixed_step,meta_inner_steps,overlap_ratio,num_tasks,max_norm,first_order,dropout,cycle
0,TQNet,96,ETTm1,0.302735,0.334017,0.330697,0.000500,0.000500,0.000250,MAE,1.0,0.0,0.0,type1,30.0,5.0,32.0,256.0,700.0,1.0,0.15,3.0,5.0,1.0,0.5,96.0
1,TQNet,192,ETTm1,0.357094,0.362497,0.359135,0.000500,0.000500,0.000250,MAE,1.0,0.0,0.0,type1,30.0,5.0,32.0,256.0,500.0,1.0,0.15,3.0,5.0,1.0,0.5,96.0
2,TQNet,336,ETTm1,0.387560,0.384858,0.381815,0.000500,0.000500,0.000250,MAE,1.0,0.0,0.0,type1,30.0,5.0,32.0,256.0,500.0,1.0,0.15,3.0,5.0,1.0,0.5,96.0
3,TQNet,720,ETTm1,0.452656,0.422947,0.419728,0.000500,0.000500,0.000250,MAE,1.0,0.0,0.0,type1,30.0,5.0,32.0,256.0,500.0,1.0,0.15,3.0,1.0,1.0,0.5,96.0
20,TQNet,Avg,ETTm1,0.375011,0.376080,0.372844,0.000500,0.000500,0.000250,NaN,1.0,0.0,0.0,NaN,30.0,5.0,32.0,256.0,550.0,1.0,0.15,3.0,4.0,1.0,0.5,96.0
4,TQNet,96,ETTm2,0.167691,0.245014,0.240380,0.001000,0.001000,0.000500,MAE,1.0,0.0,0.0,type1,30.0,5.0,32.0,256.0,700.0,1.0,0.15,3.0,5.0,1.0,0.5,96.0
5,TQNet,192,ETTm2,0.234140,0.290204,0.288442,0.000200,0.000200,0.000100,MAE,1.0,0.0,0.0,type1,30.0,5.0,32.0,256.0,500.0,1.0,0.15,3.0,5.0,1.0,0.5,96.0
6,TQNet,336,ETTm2,0.294392,0.328999,0.326632,0.000200,0.000200,0.000100,MAE,1.0,0.0,0.0,type1,30.0,5.0,32.0,256.0,700.0,1.0,0.15,3.0,5.0,1.0,0.5,96.0
7,TQNet,720,ETTm2,0.391243,0.387020,0.383325,0.000200,0.000200,0.000100,MAE,1.0,0.0,0.0,type1,30.0,5.0,32.0,256.0,700.0,1.0,0.15,3.0,5.0,1.0,0.5,96.0
21,TQNet,Avg,ETTm2,0.271867,0.312809,0.309695,0.000400,0.000400,0.000200,NaN,1.0,0.0,0.0,NaN,30.0,5.0,32.0,256.0,650.0,1.0,0.15,3.0,5.0,1.0,0.5,96.0


In [6]:
save_root = '/data/home/Licheng/workspace/ProjDF-Meta/stats_META'
df2[['model', 'pred_len', 'data_id', 'mse', 'mae', 'cov', 'learning_rate', 'meta_lr', 'num_tasks', 'meta_inner_steps', 'fixed_step']].to_csv(f'{save_root}/finetune_best_params.csv', index=False, float_format='%.4f')